# Question 3: CycleGAN - Domain Adaptation and Unpaired Image-to-Image Translation
## Sketch ↔ Photo Translation

This notebook implements CycleGAN for unpaired image-to-image translation between sketch and photo domains.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import torchvision.transforms as transforms
from torchvision.utils import save_image, make_grid
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from pathlib import Path
import glob
from collections import defaultdict
import random
from tqdm import tqdm
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'GPU Count: {torch.cuda.device_count()}')

In [ ]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f'Random seed set to {SEED}')

In [ ]:
IMAGE_SIZE = 128
BATCH_SIZE = 4
NUM_WORKERS = 2
NGF = 64
NDF = 64
RESNET_BLOCKS = 6
NUM_EPOCHS = 50
LR = 0.0002
BETAS = (0.5, 0.999)
LAMBDA_CYCLE = 10.0
LAMBDA_IDENTITY = 5.0

DATA_ROOT = '/kaggle/input/'
OUTPUT_DIR = '/kaggle/working/cyclegan_outputs/'
CHECKPOINT_DIR = '/kaggle/working/cyclegan_checkpoints/'
CKPT_INTERVAL = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Image size: {IMAGE_SIZE}x{IMAGE_SIZE}, Batch size: {BATCH_SIZE}, Epochs: {NUM_EPOCHS}')

In [ ]:
def find_sketch_and_photo_datasets():
    input_dir = DATA_ROOT
    datasets_found = {}
    if os.path.exists(input_dir):
        for folder in os.listdir(input_dir):
            folder_path = os.path.join(input_dir, folder)
            if os.path.isdir(folder_path):
                datasets_found[folder] = folder_path
                print(f'Found dataset: {folder}')
    return datasets_found

datasets = find_sketch_and_photo_datasets()
print(f'Total datasets found: {len(datasets)}')

In [ ]:
class UnpairedImageDataset(Dataset):
    def __init__(self, domain_a_path, domain_b_path, image_size=128, max_images=None):
        self.domain_a_images = self._load_images(domain_a_path, max_images)
        self.domain_b_images = self._load_images(domain_b_path, max_images)
        self.image_size = image_size
        
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size), Image.BILINEAR),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])
        
        self.transform_gray = transforms.Compose([
            transforms.Grayscale(num_output_channels=3),
            transforms.Resize((image_size, image_size), Image.BILINEAR),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])
        
        print(f'Domain A images: {len(self.domain_a_images)}, Domain B images: {len(self.domain_b_images)}')
    
    def _load_images(self, path, max_images=None):
        images = []
        if os.path.exists(path):
            for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG']:
                images.extend(glob.glob(os.path.join(path, '**', ext), recursive=True))
        if max_images and len(images) > max_images:
            images = images[:max_images]
        return list(set(images))
    
    def __len__(self):
        return max(len(self.domain_a_images), len(self.domain_b_images))
    
    def __getitem__(self, idx):
        idx_a = idx % len(self.domain_a_images)
        img_a_path = self.domain_a_images[idx_a]
        idx_b = np.random.randint(0, len(self.domain_b_images))
        img_b_path = self.domain_b_images[idx_b]
        
        try:
            img_a = Image.open(img_a_path).convert('RGB')
            img_a = self.transform_gray(img_a)
        except Exception:
            img_a = torch.ones(3, self.image_size, self.image_size)
        
        try:
            img_b = Image.open(img_b_path).convert('RGB')
            img_b = self.transform(img_b)
        except Exception:
            img_b = torch.ones(3, self.image_size, self.image_size)
        
        return {'A': img_a, 'B': img_b}

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ResidualBlock, self).__init__()
        self.conv = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=0),
            nn.InstanceNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=0),
            nn.InstanceNorm2d(out_channels)
        )
    
    def forward(self, x):
        return x + self.conv(x)

class ResNetGenerator(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, ngf=64, num_blocks=6):
        super(ResNetGenerator, self).__init__()
        self.initial = nn.Sequential(
            nn.ReflectionPad2d(3),
            nn.Conv2d(in_channels, ngf, kernel_size=7, padding=0),
            nn.InstanceNorm2d(ngf),
            nn.ReLU(inplace=True)
        )
        self.downsample = nn.Sequential(
            nn.Conv2d(ngf, ngf*2, kernel_size=3, stride=2, padding=1),
            nn.InstanceNorm2d(ngf*2),
            nn.ReLU(inplace=True),
            nn.Conv2d(ngf*2, ngf*4, kernel_size=3, stride=2, padding=1),
            nn.InstanceNorm2d(ngf*4),
            nn.ReLU(inplace=True)
        )
        residual_blocks = [ResidualBlock(ngf*4, ngf*4) for _ in range(num_blocks)]
        self.residual = nn.Sequential(*residual_blocks)
        self.upsample = nn.Sequential(
            nn.ConvTranspose2d(ngf*4, ngf*2, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.InstanceNorm2d(ngf*2),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(ngf*2, ngf, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.InstanceNorm2d(ngf),
            nn.ReLU(inplace=True)
        )
        self.final = nn.Sequential(
            nn.ReflectionPad2d(3),
            nn.Conv2d(ngf, out_channels, kernel_size=7, padding=0),
            nn.Tanh()
        )
    
    def forward(self, x):
        x = self.initial(x)
        x = self.downsample(x)
        x = self.residual(x)
        x = self.upsample(x)
        x = self.final(x)
        return x

In [ ]:
class PatchGANDiscriminator(nn.Module):
    def __init__(self, in_channels=3, ndf=64):
        super(PatchGANDiscriminator, self).__init__()
        def conv_block(in_ch, out_ch, normalize=True):
            layers = [nn.Conv2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1)]
            if normalize:
                layers.append(nn.InstanceNorm2d(out_ch))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return nn.Sequential(*layers)
        
        self.model = nn.Sequential(
            conv_block(in_channels, ndf, normalize=False),
            conv_block(ndf, ndf*2),
            conv_block(ndf*2, ndf*4),
            nn.Conv2d(ndf*4, 1, kernel_size=4, stride=1, padding=1)
        )
    
    def forward(self, x):
        return self.model(x)

In [ ]:
class CycleGANLoss(nn.Module):
    def __init__(self, lambda_cycle=10.0, lambda_identity=5.0):
        super(CycleGANLoss, self).__init__()
        self.lambda_cycle = lambda_cycle
        self.lambda_identity = lambda_identity
        self.criterion_gan = nn.MSELoss()
        self.criterion_cycle = nn.L1Loss()
        self.criterion_identity = nn.L1Loss()
    
    def adversarial_loss(self, disc_output, is_real):
        target = torch.ones_like(disc_output) if is_real else torch.zeros_like(disc_output)
        return self.criterion_gan(disc_output, target)
    
    def cycle_loss(self, real_x, reconstructed_x):
        return self.criterion_cycle(real_x, reconstructed_x) * self.lambda_cycle
    
    def identity_loss(self, real_x, identical_x):
        return self.criterion_identity(real_x, identical_x) * self.lambda_identity

In [ ]:
G_AB = ResNetGenerator(in_channels=3, out_channels=3, ngf=NGF, num_blocks=RESNET_BLOCKS).to(device)
G_BA = ResNetGenerator(in_channels=3, out_channels=3, ngf=NGF, num_blocks=RESNET_BLOCKS).to(device)
D_A = PatchGANDiscriminator(in_channels=3, ndf=NDF).to(device)
D_B = PatchGANDiscriminator(in_channels=3, ndf=NDF).to(device)

if torch.cuda.device_count() > 1:
    G_AB = nn.DataParallel(G_AB)
    G_BA = nn.DataParallel(G_BA)
    D_A = nn.DataParallel(D_A)
    D_B = nn.DataParallel(D_B)
    print(f'Using {torch.cuda.device_count()} GPUs')

optimizer_G = optim.Adam(list(G_AB.parameters()) + list(G_BA.parameters()), lr=LR, betas=BETAS)
optimizer_D_A = optim.Adam(D_A.parameters(), lr=LR, betas=BETAS)
optimizer_D_B = optim.Adam(D_B.parameters(), lr=LR, betas=BETAS)
cycle_loss = CycleGANLoss(lambda_cycle=LAMBDA_CYCLE, lambda_identity=LAMBDA_IDENTITY)
scaler = GradScaler()

print('Models initialized')

In [ ]:
class ImageBuffer:
    def __init__(self, max_size=50):
        self.max_size = max_size
        self.images = []
    
    def push_and_pop(self, images):
        if self.max_size == 0:
            return images
        returned_images = []
        for image in images:
            if len(self.images) < self.max_size:
                self.images.append(image.clone().detach())
                returned_images.append(image)
            else:
                if random.uniform(0, 1) > 0.5:
                    idx = random.randint(0, self.max_size - 1)
                    temp = self.images[idx].clone()
                    self.images[idx] = image.clone().detach()
                    returned_images.append(temp)
                else:
                    returned_images.append(image)
        return torch.cat(returned_images, dim=0)

fake_A_buffer = ImageBuffer()
fake_B_buffer = ImageBuffer()

In [ ]:
def train_cyclegan(train_loader, num_epochs):
    G_AB.train()
    G_BA.train()
    D_A.train()
    D_B.train()
    
    losses_history = {'G': [], 'D_A': [], 'D_B': [], 'Cycle': [], 'Identity': [], 'SSIM': [], 'PSNR': []}
    
    for epoch in range(num_epochs):
        epoch_losses = defaultdict(float)
        for batch_idx, batch in enumerate(tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')):
            real_A = batch['A'].to(device)
            real_B = batch['B'].to(device)
            
            optimizer_G.zero_grad()
            with autocast():
                fake_B = G_AB(real_A)
                fake_A = G_BA(real_B)
                reconstructed_A = G_BA(fake_B)
                reconstructed_B = G_AB(fake_A)
                identity_A = G_BA(real_A)
                identity_B = G_AB(real_B)
                
                loss_G_AB = cycle_loss.adversarial_loss(D_B(fake_B), True)
                loss_G_BA = cycle_loss.adversarial_loss(D_A(fake_A), True)
                loss_cycle_A = cycle_loss.cycle_loss(real_A, reconstructed_A)
                loss_cycle_B = cycle_loss.cycle_loss(real_B, reconstructed_B)
                loss_identity_A = cycle_loss.identity_loss(real_A, identity_A)
                loss_identity_B = cycle_loss.identity_loss(real_B, identity_B)
                loss_G = loss_G_AB + loss_G_BA + loss_cycle_A + loss_cycle_B + loss_identity_A + loss_identity_B
            
            scaler.scale(loss_G).backward()
            scaler.step(optimizer_G)
            scaler.update()
            
            optimizer_D_A.zero_grad()
            optimizer_D_B.zero_grad()
            
            with autocast():
                fake_A_detached = fake_A_buffer.push_and_pop(fake_A.detach())
                loss_D_A_real = cycle_loss.adversarial_loss(D_A(real_A), True)
                loss_D_A_fake = cycle_loss.adversarial_loss(D_A(fake_A_detached), False)
                loss_D_A = (loss_D_A_real + loss_D_A_fake) * 0.5
                
                fake_B_detached = fake_B_buffer.push_and_pop(fake_B.detach())
                loss_D_B_real = cycle_loss.adversarial_loss(D_B(real_B), True)
                loss_D_B_fake = cycle_loss.adversarial_loss(D_B(fake_B_detached), False)
                loss_D_B = (loss_D_B_real + loss_D_B_fake) * 0.5
            
            scaler.scale(loss_D_A + loss_D_B).backward()
            scaler.step(optimizer_D_A)
            scaler.step(optimizer_D_B)
            scaler.update()
            
            epoch_losses['G'] += loss_G.item()
            epoch_losses['D_A'] += loss_D_A.item()
            epoch_losses['D_B'] += loss_D_B.item()
            epoch_losses['Cycle'] += (loss_cycle_A + loss_cycle_B).item()
            epoch_losses['Identity'] += (loss_identity_A + loss_identity_B).item()
            
            # Calculate metrics for current batch
            with torch.no_grad():
                batch_ssim, batch_psnr = calculate_metrics(real_A, identity_A) # Self-identity as proxy or use reconstructed
                epoch_losses['SSIM'] += batch_ssim
                epoch_losses['PSNR'] += batch_psnr
        
        num_batches = len(train_loader)
        for key in epoch_losses:
            epoch_losses[key] /= num_batches
            losses_history[key].append(epoch_losses[key])
        
        print(f"Epoch {epoch+1}: G={epoch_losses['G']:.4f}, D_A={epoch_losses['D_A']:.4f}, D_B={epoch_losses['D_B']:.4f}")
        
        if (epoch + 1) % CKPT_INTERVAL == 0:
            torch.save({'G_AB': G_AB.state_dict(), 'G_BA': G_BA.state_dict()}, 
                       os.path.join(CHECKPOINT_DIR, f'cyclegan_epoch{epoch+1}.pth'))
    
    return losses_history

In [ ]:
def plot_losses(losses_history):
    fig, axes = plt.subplots(3, 3, figsize=(15, 12))
    fig.suptitle('CycleGAN Training Losses', fontsize=16)
    axes = axes.flatten()
    
    loss_names = ['G', 'D_A', 'D_B', 'Cycle', 'Identity', 'SSIM', 'PSNR']
    for idx, name in enumerate(loss_names):
        if name in losses_history:
            axes[idx].plot(losses_history[name], linewidth=2)
            axes[idx].set_title(f'{name} Loss')
            axes[idx].set_xlabel('Epoch')
            axes[idx].set_ylabel('Loss')
            axes[idx].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'cyclegan_losses.png'), dpi=100, bbox_inches='tight')
    plt.show()

In [ ]:
def generate_inference_images(model_type='G_AB', num_images=10):
    G_AB.eval()
    G_BA.eval()
    with torch.no_grad():
        z = torch.randn(num_images, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)
        if model_type == 'G_AB':
            output = G_AB(z)
        else:
            output = G_BA(z)
        grid = make_grid(output, nrow=5, normalize=True)
    img_array = grid.cpu().numpy().transpose(1, 2, 0)
    img_array = (img_array + 1) / 2
    return np.clip(img_array, 0, 1)

In [ ]:
def calculate_metrics(real, generated):
    """Calculate SSIM and PSNR between two batches of images."""
    ssim_values = []
    psnr_values = []
    real_np = (real.detach().cpu().numpy() + 1) / 2
    generated_np = (generated.detach().cpu().numpy() + 1) / 2
    for i in range(real_np.shape[0]):
        r = real_np[i].transpose(1, 2, 0)
        g = generated_np[i].transpose(1, 2, 0)
        s = ssim(r, g, multichannel=True, data_range=1.0, channel_axis=2)
        p = psnr(r, g, data_range=1.0)
        ssim_values.append(s)
        psnr_values.append(p)
    return np.mean(ssim_values), np.mean(psnr_values)

def visualize_results(model_ab, model_ba, test_loader, num_samples=5):
    model_ab.eval()
    model_ba.eval()
    with torch.no_grad():
        batch = next(iter(test_loader))
        real_A = batch['A'][:num_samples].to(device)
        fake_B = model_ab(real_A)
        rec_A = model_ba(fake_B)
        imgs = torch.cat([real_A, fake_B, rec_A], dim=0)
        grid = make_grid(imgs, nrow=num_samples, normalize=True)
        plt.figure(figsize=(15, 10))
        plt.imshow(grid.cpu().numpy().transpose(1, 2, 0))
        plt.title("Cycle Translation: Top (Input Sketch), Middle (Gen Photo), Bottom (Rec Sketch)")
        plt.axis('off')
        plt.show()

domain_a_path = None
domain_b_path = None

if 'train_loader' in locals() and train_loader is not None:
    visualize_results(G_AB, G_BA, train_loader, num_samples=5)

for dataset_name, dataset_path in datasets.items():
    if any(keyword in dataset_name.lower() for keyword in ['sketch', 'quickdraw', 'tu-berlin']):
        domain_a_path = dataset_path
    elif any(keyword in dataset_name.lower() for keyword in ['photo', 'image', 'face']):
        domain_b_path = dataset_path

if domain_a_path is None or domain_b_path is None:
    domain_a_path = os.path.join(DATA_ROOT, 'sketch_dataset')
    domain_b_path = os.path.join(DATA_ROOT, 'photo_dataset')

print(f'Domain A (Sketches): {domain_a_path}')
print(f'Domain B (Photos): {domain_b_path}')

if os.path.exists(domain_a_path) and os.path.exists(domain_b_path):
    train_dataset = UnpairedImageDataset(domain_a_path, domain_b_path, image_size=IMAGE_SIZE, max_images=5000)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    print(f'DataLoader created with {len(train_dataset)} images')
else:
    print('Datasets not found')
    train_loader = None

In [ ]:
if train_loader is not None:
    print('Starting CycleGAN training...')
    losses_history = train_cyclegan(train_loader, NUM_EPOCHS)
    print('Training completed!')
else:
    print('Cannot start training: DataLoader not initialized')

In [ ]:
torch.save({'G_AB': G_AB.state_dict(), 'G_BA': G_BA.state_dict(), 'D_A': D_A.state_dict(), 'D_B': D_B.state_dict()}, 
           os.path.join(CHECKPOINT_DIR, 'cyclegan_final.pth'))
print('Final models saved')

In [ ]:
if 'losses_history' in locals():
    plot_losses(losses_history)
else:
    print('No training history available')

In [ ]:
print('='*60)
print('CycleGAN Training Summary')
print('='*60)
print(f'Device: {device}')
print(f'Image Size: {IMAGE_SIZE}x{IMAGE_SIZE}')
print(f'Epochs: {NUM_EPOCHS}, Batch Size: {BATCH_SIZE}')
print(f'ResNet Blocks: {RESNET_BLOCKS}')
print(f'Lambda Cycle: {LAMBDA_CYCLE}, Lambda Identity: {LAMBDA_IDENTITY}')
print(f'Output Dir: {OUTPUT_DIR}')
print(f'Checkpoint Dir: {CHECKPOINT_DIR}')
print('='*60)